![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# M3L2 E19 - Introducción a LangGraph: estado, checkpointer y persistencia (Resolution)

## Qué es este notebook

`RunnableWithMessageHistory` (E04, E18) alcanza para un chatbot lineal. Pero cuando aparecen agentes con herramientas, ciclos, reintentos o múltiples pasos, la forma recomendada de manejar estado y persistencia pasa a ser **LangGraph**. Este notebook es la primera introducción del módulo a LangGraph: un grafo mínimo, el concepto de `thread_id`, y cómo un `checkpointer` guarda el estado entre invocaciones — con `InMemorySaver`, que funciona sin ninguna base de datos externa.


In [ ]:
# langgraph no es una dependencia de langchain-openai: hay que instalarlo aparte.
# %pip (no !pip): instala en el Python del kernel activo, no depende del PATH de tu shell.
%pip install langgraph langchain-openai
# Opcional (BLOQUE 6, SqliteSaver): %pip install langgraph-checkpoint-sqlite


In [ ]:
import os, getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


## BLOQUE 1 — ¿Cuándo conviene LangGraph?

| `RunnableWithMessageHistory` (E04, E18) | LangGraph |
|---|---|
| Chatbot básico, preguntas y respuestas | Agentes con herramientas |
| RAG conversacional simple | Múltiples nodos y decisiones condicionales |
| Flujo lineal | Ciclos, reintentos controlados |
| — | Human-in-the-loop |
| — | Ejecución durable de larga duración |
| — | Multiagentes |

LangGraph no reemplaza al modelo ni al servidor de inferencia — es un runtime de orquestación de **más bajo nivel** que una chain simple, pensado para flujos con estado explícito.


## BLOQUE 2 — Un grafo mínimo con `MessagesState`

```text
START -> assistant_node -> END
```

`MessagesState` es un estado predefinido de LangGraph que ya sabe acumular una lista de mensajes — el equivalente al historial que armábamos a mano en E04.


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def assistant_node(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant_node)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)

graph_sin_memoria = builder.compile()

resultado = graph_sin_memoria.invoke({
    "messages": [{"role": "user", "content": "Explica que es un estado en LangGraph, en una oracion."}]
})

print(resultado["messages"][-1].content)


> Punto clave: `assistant_node` no sabe nada de LM Studio, Ollama ni OpenAI — solo necesita un objeto con `.invoke(messages)`. El mismo `model_factory` de E17 encajaría acá sin cambios.


## BLOQUE 3 — Sin checkpointer, este grafo tampoco recuerda

In [ ]:
resultado_1 = graph_sin_memoria.invoke({
    "messages": [{"role": "user", "content": "Me llamo Lucia y soy analista de datos."}]
})
resultado_2 = graph_sin_memoria.invoke({
    "messages": [{"role": "user", "content": "Como me llamo y cual es mi profesion?"}]
})

print(f"Respuesta 1: {resultado_1['messages'][-1].content}")
print(f"Respuesta 2: {resultado_2['messages'][-1].content}")
print()
print("Cada .invoke() es un grafo 'nuevo': sin checkpointer, no hay memoria entre llamadas.")


## BLOQUE 4 — `checkpointer` y `thread_id`

Un **checkpointer** guarda una foto (snapshot) del estado del grafo después de cada paso, identificada por un `thread_id` — el equivalente al `session_id` de `RunnableWithMessageHistory`.

```python
config = {"configurable": {"thread_id": conversation_id}}
```

| Sistema | Alcance | Guarda |
|---|---|---|
| **Checkpointer** | Un solo `thread` | Mensajes, resultados de tools, posición del flujo — memoria de corto plazo |
| **Store** | Entre threads | Preferencias, perfiles, hechos del usuario — memoria de largo plazo |

```text
SQLChatMessageHistory (E18)
  -> persiste mensajes de una CHAIN.

Checkpointer de LangGraph
  -> persiste snapshots del ESTADO de un GRAFO (mensajes + lo que necesite el flujo).
```


## BLOQUE 5 — `InMemorySaver`: el checkpointer más simple

`InMemorySaver` guarda el estado en RAM — se pierde al reiniciar el proceso, pero no necesita ninguna base de datos externa. Es el equivalente en LangGraph al diccionario `historiales` que usamos en E04.


In [ ]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph_con_memoria = builder.compile(checkpointer=checkpointer)

conversacion_1 = str(uuid.uuid4())
config_1 = {"configurable": {"thread_id": conversacion_1}}

graph_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "Me llamo Lucia y soy analista de datos."}]},
    config=config_1,
)
resultado = graph_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "Como me llamo y cual es mi profesion?"}]},
    config=config_1,
)

print(f"thread_id: {conversacion_1}")
print(f"Respuesta: {resultado['messages'][-1].content}")


In [ ]:
# Aislamiento: un thread_id distinto no comparte estado
conversacion_2 = str(uuid.uuid4())
config_2 = {"configurable": {"thread_id": conversacion_2}}

resultado_otro_thread = graph_con_memoria.invoke(
    {"messages": [{"role": "user", "content": "Como me llamo?"}]},
    config=config_2,
)

print(f"thread_id: {conversacion_2} (nunca hablo con este thread antes)")
print(f"Respuesta: {resultado_otro_thread['messages'][-1].content}")
print()
print("El grafo no tiene informacion de Lucia en este thread_id: son estados completamente separados.")


In [ ]:
# Inspeccionar el estado guardado por el checkpointer
estado_guardado = graph_con_memoria.get_state(config_1)
print(f"Mensajes en el estado del thread {conversacion_1}: {len(estado_guardado.values['messages'])}")
for m in estado_guardado.values["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")


## BLOQUE 6 — Checkpointers persistentes: `SqliteSaver` y `PostgresSaver`

```text
InMemorySaver -> pruebas rapidas; se pierde todo al reiniciar el proceso.
SqliteSaver   -> desarrollo local durable (archivo en disco).
PostgresSaver -> produccion y concurrencia.
```

La celda de abajo es **opcional**: usa `SqliteSaver` si el paquete `langgraph-checkpoint-sqlite` está instalado; si no, se saltea con un mensaje. La lógica del grafo (`builder`, `assistant_node`) es exactamente la misma — solo cambia el checkpointer.


In [ ]:
try:
    from langgraph.checkpoint.sqlite import SqliteSaver

    with SqliteSaver.from_conn_string("m3l2_e19_checkpoints.db") as sqlite_checkpointer:
        graph_sqlite = builder.compile(checkpointer=sqlite_checkpointer)

        thread_sqlite = str(uuid.uuid4())
        cfg_sqlite = {"configurable": {"thread_id": thread_sqlite}}

        graph_sqlite.invoke(
            {"messages": [{"role": "user", "content": "Mi color favorito es el azul."}]},
            config=cfg_sqlite,
        )
        r = graph_sqlite.invoke(
            {"messages": [{"role": "user", "content": "Cual es mi color favorito?"}]},
            config=cfg_sqlite,
        )
        print(f"SqliteSaver -> {r['messages'][-1].content}")
except ImportError:
    print("Salteado: falta instalar langgraph-checkpoint-sqlite (pip install langgraph-checkpoint-sqlite).")


En producción, `PostgresSaver` sigue el mismo patrón, apuntando a una base real:

```python
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DATABASE_URL) as checkpointer:
    checkpointer.setup()  # una sola vez, al inicializar la base
    graph = builder.compile(checkpointer=checkpointer)
    graph.invoke({"messages": [...]}, config={"configurable": {"thread_id": conversation_id}})
```

`checkpointer.setup()` crea las tablas internas de LangGraph. En producción conviene ejecutarlo como migración o script de despliegue, no en cada arranque de la aplicación.


## BLOQUE 7 — Arquitectura recomendada

```text
Frontend
   ↓
API (FastAPI)
   ↓
Servicio de chat
   ↓
LangChain o LangGraph
   ↓                    ↓
Modelo (Ollama/OpenAI)  PostgreSQL (checkpoints + tablas de negocio)
```

| Escenario | Recomendación |
|---|---|
| Chatbot lineal pequeño | `RunnableWithMessageHistory` + `SQLChatMessageHistory` (E18) |
| RAG conversacional simple | `RunnableWithMessageHistory` + ventana o resumen |
| Agente con herramientas | LangGraph + `PostgresSaver` + `thread_id` |
| Memoria entre conversaciones | `Store` o tablas propias, con `user_id` como namespace |
| Producto multiusuario | `users` + `conversations` + `messages` + permisos + PostgreSQL |


## Resumen — Lo que demuestra E19

```text
StateGraph        = grafo de nodos con estado explicito
MessagesState     = estado predefinido que acumula una lista de mensajes
thread_id         = identificador de conversacion (equivalente a session_id)
checkpointer      = guarda snapshots del estado por thread_id (memoria de corto plazo)
Store             = memoria compartida ENTRE threads (largo plazo)
```

| E04 / E18 | E19 |
|---|---|
| `session_id` | `thread_id` |
| `RunnableWithMessageHistory` | `graph.compile(checkpointer=...)` |
| `InMemoryChatMessageHistory` / `SQLChatMessageHistory` | `InMemorySaver` / `SqliteSaver` / `PostgresSaver` |
| Chain lineal | Grafo con nodos, condiciones y ciclos |

**Relacionado con**: E04 - Chat con memoria, E06 - Agente completo, E18 - Persistencia SQL.


## Checks automáticos

In [ ]:
def run_checks():
    estado = graph_con_memoria.get_state(config_1)
    assert len(estado.values["messages"]) >= 4

    r_mismo_thread = graph_con_memoria.invoke(
        {"messages": [{"role": "user", "content": "Recorda mi profesion?"}]},
        config=config_1,
    )
    assert "analista" in r_mismo_thread["messages"][-1].content.lower() or "datos" in r_mismo_thread["messages"][-1].content.lower()

    estado_thread_2 = graph_con_memoria.get_state(config_2)
    contenido_thread_2 = " ".join(m.content for m in estado_thread_2.values["messages"])
    assert "lucia" not in contenido_thread_2.lower()

    print("M3L2 E19 Resolution checks passed")


run_checks()


In [ ]:
# Limpieza: borrar el archivo de checkpoints de SQLite si se llego a crear
import os as _os

if _os.path.exists("m3l2_e19_checkpoints.db"):
    _os.remove("m3l2_e19_checkpoints.db")
    print("m3l2_e19_checkpoints.db eliminado.")


## Referencias oficiales

- [LangGraph — Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory)
- [LangGraph — Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
